# Ranking Signal Analysis

Reproducible capstone runner. The numbered Python modules beside this notebook contain the reviewed implementation; generated content-level files remain outside Git. All findings are observational and intended for decision support.

## 1. Public-safe schema discovery
Reuse the committed metadata receipt when present. A fresh checkout can regenerate it with `HF_TOKEN`; no raw rows are printed.

In [1]:
from pathlib import Path
schema_receipt = Path('../../docs/schema_notes.md')
if schema_receipt.exists():
    print(f'Using committed public-safe schema receipt: {schema_receipt.as_posix()}')
else:
    get_ipython().run_line_magic('run', './00_schema_discovery.py')

Using committed public-safe schema receipt: ../../docs/schema_notes.md


## 2. Leakage-separated feature windows
Reuse ignored content-level feature caches when present; a fresh checkout rebuilds the May robustness and June primary windows from the gated warehouse.

In [2]:
import pyarrow.parquet as pq
feature_dir = Path('../../outputs/data')
feature_paths = [
    feature_dir / 'features_2026_04_30.parquet',
    feature_dir / 'features_2026_05_31.parquet',
]
for cutoff, path in [('2026-04-30', feature_paths[0]), ('2026-05-31', feature_paths[1])]:
    if path.exists():
        print(f'Cache hit: {path.name} | rows={pq.ParquetFile(path).metadata.num_rows:,}')
    else:
        get_ipython().run_line_magic('run', f'./01_signal_features_audited.py --cutoff {cutoff}')

Cache hit: features_2026_04_30.parquet | rows=132,875
Cache hit: features_2026_05_31.parquet | rows=130,833


## 3. Grouped modeling and robustness
The same held-out-client rule is used for baseline, Ridge, and random forest comparisons.

In [3]:
%run ./02_modeling.py

{
  "rows": 130833,
  "train_rows": 120612,
  "test_rows": 10221,
  "train_clients": 39,
  "test_clients": 9,
  "feature_cutoff": "2026-05-31",
  "target_start": "2026-06-01",
  "target_end": "2026-06-28",
  "metrics": {
    "baseline": {
      "r2": -0.049131612765582755,
      "rmse": 10.098785757223997,
      "spearman": -0.0964843057831924
    },
    "ridge": {
      "r2": -0.004792692816643518,
      "rmse": 9.883082168795072,
      "spearman": 0.1652579319313056
    },
    "random_forest": {
      "r2": -0.07451745105034946,
      "rmse": 10.22023563047264,
      "spearman": 0.1173919831750979
    },
    "permutation_importance": {
      "avg_position_28d": 0.06280154675359184,
      "position_volatility_90d": 0.02039839241800112,
      "position_trend_slope_90d": -0.013190782023263884,
      "history_impressions_90d": -0.010451484393169785,
      "history_ctr_90d": -0.006058900337723827,
      "history_ctr_28d": 0.002733676210646611,
      "ctr_gap": 0.0008439972089608894,
     

## 4. Descriptive query analysis and paper artifacts
Query signals overlap the primary outcome window and are explicitly excluded from the predictive models.

In [4]:
%run ./03_query_associations.py
%run ./04_analysis_outputs.py

                           signal      n  spearman_with_position_change                            interpretation
     query_impression_entropy_90d 111928                       0.144823 descriptive_shared_window_not_model_input
     visible_query_click_momentum 111928                      -0.071561 descriptive_shared_window_not_model_input
visible_query_impression_momentum 111928                      -0.087455 descriptive_shared_window_not_model_input


{
  "rows": 130833,
  "clients": 48,
  "feature_window_days": 90,
  "outcome_window_days": 28,
  "median_position_change": 0.034,
  "archetypes": [
    {
      "name": "CTR opportunity",
      "reason_code": "CTR-GAP-LOW",
      "count": 13165,
      "share_pct": 10.1,
      "median_position_change": 0.23
    },
    {
      "name": "Volatile visibility",
      "reason_code": "POS-VOL-HIGH",
      "count": 13084,
      "share_pct": 10.0,
      "median_position_change": 3.29
    },
    {
      "name": "Deteriorating trend",
      "reason_code": "POS-TREND-DOWN",
      "count": 13084,
      "share_pct": 10.0,
      "median_position_change": 0.25
    }
  ]
}
